# Pre-training Hebrew TrOCR on synthetic data (ViT encoder + DictaBERT decoder)

Stage 1 of the recipe: **synthetic pretrain -> human finetune**. Mirrors
`train_trocr_hebrew.ipynb` but trains on the synthetic line dataset and
**checkpoints to the HuggingFace Hub** so a lost Vertex / Colab Enterprise session does not wipe progress.

**Target hardware:** single L4 (24 GB, bf16). On a Turing card (RTX 2080) set
`PRECISION = "fp16"` below.

> After this finishes, feed the saved model into the human-finetune notebook as
> its `MODEL_ID` -- never the other way round (synthetic-after-human causes
> catastrophic forgetting).

## 1. Setup

Self-contained: downloads the model + dataset from HuggingFace. Run top to bottom.

In [ ]:
# Run once on a fresh VM. Comment out if deps are already installed.
!pip install -q torch transformers datasets accelerate jiwer pillow matplotlib

In [ ]:
import os
import torch
import jiwer
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    VisionEncoderDecoderModel,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print("gpu   :", torch.cuda.get_device_name(0))
    print("vram  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
    print("bf16  :", torch.cuda.is_bf16_supported())

In [ ]:
# --- HuggingFace auth (Vertex / Colab Enterprise: no Drive -> we checkpoint to the Hub) ---
# Needs a WRITE token, entered EVERY session (box/CLI auth does NOT carry to Vertex).
# Create one at https://huggingface.co/settings/tokens  (token type: Write)
from huggingface_hub import notebook_login, whoami
notebook_login()
print("logged in as:", whoami()["name"])

In [ ]:
# --- HebrewBlockProcessor (inlined so this notebook is self-contained) ---
from PIL import Image, ImageOps

class HebrewBlockProcessor:
    """Mirror (RTL->LTR) -> resize to 64px height -> tile into a 384x384 ViT container."""
    TARGET_HEIGHT = 64
    CONTAINER_SIZE = 384
    IMAGE_MEAN = [0.5, 0.5, 0.5]
    IMAGE_STD = [0.5, 0.5, 0.5]

    def __call__(self, images, return_tensors="pt"):
        if not isinstance(images, list):
            images = [images]
        pixel_values = torch.stack([self._process(img) for img in images])
        return {"pixel_values": pixel_values}

    def _process(self, image):
        image = image.convert("RGB")
        image = ImageOps.mirror(image)
        w, h = image.size
        new_w = max(1, round(w * self.TARGET_HEIGHT / h))
        image = image.resize((new_w, self.TARGET_HEIGHT), Image.LANCZOS)
        container = Image.new("RGB", (self.CONTAINER_SIZE, self.CONTAINER_SIZE), (255, 255, 255))
        img_arr = np.array(image)
        src_x, dest_x, dest_y = 0, 0, 0
        while src_x < new_w and dest_y < self.CONTAINER_SIZE:
            chunk_w = min(new_w - src_x, self.CONTAINER_SIZE - dest_x)
            chunk = Image.fromarray(img_arr[:, src_x:src_x + chunk_w])
            container.paste(chunk, (dest_x, dest_y))
            src_x += chunk_w
            dest_x += chunk_w
            if dest_x >= self.CONTAINER_SIZE:
                dest_x = 0
                dest_y += self.TARGET_HEIGHT
        t = torch.tensor(np.array(container), dtype=torch.float32).permute(2, 0, 1) / 255.0
        mean = torch.tensor(self.IMAGE_MEAN).view(3, 1, 1)
        std = torch.tensor(self.IMAGE_STD).view(3, 1, 1)
        return (t - mean) / std

## 2. Config

For a fast first experiment keep `ENCODER_FROZEN = True`. The crash-resilience
knobs at the bottom control how often checkpoints are written to Drive.

In [ ]:
MODEL_ID          = "cyttic/trocr-hebrew-untrained"
DATASET_ID        = "cyttic/trocr-hebrew-synthetic"

EPOCHS            = 1        # one pretrain pass; warm-start for more epochs later
BATCH_SIZE        = 8        # see ENCODER_FROZEN note below
GRAD_ACCUM        = 1
LR                = 5e-5     # if an unfrozen encoder degrades, try 2e-5 / 3e-5
MAX_TARGET_LENGTH = 128
NUM_WORKERS       = 4
PRECISION         = "bf16"   # "bf16" (L4) | "fp16" (RTX 2080) | "no"
MAX_STEPS         = -1       # set e.g. 50 for a quick smoke test

# --- encoder freeze ---
# True  : ViT encoder frozen, only decoder + cross-attention train (less VRAM, but
#         the random cross-attention must align to FIXED features -> slow convergence,
#         which is the CER-0.70-after-1-epoch curve we saw).
# False : train the WHOLE model incl. the encoder. Usually converges much faster on
#         the synthetic visual->text bridge, but uses more VRAM -> keep BATCH_SIZE
#         around 8 on the L4 (try 12 if you have headroom; drop to 4 if you OOM).
ENCODER_FROZEN    = False

# Per-variant names so frozen and unfrozen runs never collide.
RUN_NAME       = "trocr-hebrew-synthetic" + ("" if ENCODER_FROZEN else "-unfrozen")
OUTPUT_DIR     = f"output/{RUN_NAME}"          # local staging on the (ephemeral) Vertex VM
HUB_CKPTS_REPO = f"cyttic/{RUN_NAME}-ckpts"    # durable: latest checkpoint pushed here every save
print("variant      :", "frozen encoder" if ENCODER_FROZEN else "FULL (encoder unfrozen)")
print("local staging:", OUTPUT_DIR)
print("hub ckpts    :", HUB_CKPTS_REPO)

# --- crash-resilience / eval-cost knobs ---
SAVE_STEPS  = 2000   # checkpoint (push to Hub) every N steps
EVAL_STEPS  = 5000   # evaluate every N steps
EVAL_SUBSET = 2000   # eval on a subset (full 125k test is too slow per-eval)
SAVE_LIMIT  = 3      # keep only the last N local checkpoints

## 3. Model, tokenizer, processor

In [ ]:
model     = VisionEncoderDecoderModel.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
processor = HebrewBlockProcessor()

# generation_config takes priority over model.config in recent transformers,
# so set the special tokens on it explicitly or generate() fails during eval.
model.generation_config.decoder_start_token_id = tokenizer.cls_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.sep_token_id
model.generation_config.max_new_tokens = None

if ENCODER_FROZEN:
    for p in model.encoder.parameters():
        p.requires_grad = False

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"params total    : {n_total/1e6:.1f}M")
print(f"params trainable: {n_train/1e6:.1f}M")

## 4. Dataset

In [ ]:
ds = load_dataset(DATASET_ID)
print(ds)

# Full 125k test set is too slow to generate on every eval -> use a fixed subset.
eval_ds = ds["test"].select(range(min(EVAL_SUBSET, len(ds["test"]))))
print("eval subset:", len(eval_ds))

## 5. Sanity checks

The synthetic set has **no `source_doc`** column. Instead confirm no transcript
is shared between train and test (leakage check), then *see* what
`HebrewBlockProcessor` does: mirror (RTL->LTR) -> 64px -> tile into 384x384.

In [ ]:
tr = set(ds["train"]["text"])
te = set(ds["test"]["text"])
print(f"train texts: {len(tr)} | test texts: {len(te)} | OVERLAP: {len(tr & te)} (should be ~0)")

In [ ]:
sample = ds["train"][1]
img = sample["image"].convert("RGB")
print("text:", sample["text"])

pv = processor([img])["pixel_values"][0]
shown = (pv * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].imshow(img);   ax[0].set_title("raw synthetic line (64px high)"); ax[0].axis("off")
ax[1].imshow(shown); ax[1].set_title("after HebrewBlockProcessor (mirrored + tiled 384x384)"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## 6. Collator + metrics (CER / WER)

In [ ]:
def collate(batch):
    images = [ex["image"].convert("RGB") for ex in batch]
    texts  = [ex["text"] for ex in batch]
    pixel_values = processor(images)["pixel_values"]
    labels = tokenizer(
        texts, padding="longest", truncation=True,
        max_length=MAX_TARGET_LENGTH, return_tensors="pt",
    ).input_ids
    labels[labels == tokenizer.pad_token_id] = -100
    return {"pixel_values": pixel_values, "labels": labels}


def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    pred_ids  = np.where(pred_ids  < 0, tokenizer.pad_token_id, pred_ids)
    label_ids = np.where(label_ids < 0, tokenizer.pad_token_id, label_ids)
    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"cer": jiwer.cer(label_str, pred_str),
            "wer": jiwer.wer(label_str, pred_str)}

## 7. Train (checkpoints to the Hub, auto-resume)

Checkpoints are pushed to `HUB_CKPTS_REPO` on the Hub every `SAVE_STEPS`. **If the
session dies, just re-run the notebook top-to-bottom** -- the train cell finds
the last checkpoint and resumes from it.

In [ ]:
targs = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS,
    bf16=(PRECISION == "bf16"),
    fp16=(PRECISION == "fp16"),
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=1,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    logging_steps=50,
    save_total_limit=SAVE_LIMIT,
    load_best_model_at_end=False,   # pretrain: keep latest, not "best by subset CER"
    dataloader_num_workers=NUM_WORKERS,
    remove_unused_columns=False,    # keep image/text for the custom collator
    report_to="none",
    push_to_hub=True,
    hub_model_id=HUB_CKPTS_REPO,
    hub_strategy="checkpoint",     # only the latest checkpoint kept on the Hub (for resume)
    hub_private_repo=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=targs,
    train_dataset=ds["train"],
    eval_dataset=eval_ds,
    data_collator=collate,
    compute_metrics=compute_metrics,
)

In [ ]:
# Resume after a lost session. Vertex has no Drive, so the latest checkpoint lives on
# the Hub (hub_strategy="checkpoint" -> a "last-checkpoint" subfolder). Pull it back.
from huggingface_hub import snapshot_download
from huggingface_hub.utils import RepositoryNotFoundError

last_ckpt = get_last_checkpoint(OUTPUT_DIR) if os.path.isdir(OUTPUT_DIR) else None
if last_ckpt is None:
    try:
        snapshot_download(HUB_CKPTS_REPO, repo_type="model",
                          local_dir=OUTPUT_DIR, allow_patterns="last-checkpoint/*")
        cand = os.path.join(OUTPUT_DIR, "last-checkpoint")
        last_ckpt = cand if os.path.isdir(cand) else None
    except RepositoryNotFoundError:
        pass  # first run -- ckpts repo does not exist yet

print("Resuming from", last_ckpt) if last_ckpt else print("No checkpoint -- training from scratch")
trainer.train(resume_from_checkpoint=last_ckpt)

## 8. Final evaluation (beam search) on the eval subset

In [ ]:
metrics = trainer.evaluate(eval_dataset=eval_ds, num_beams=4,
                           max_length=MAX_TARGET_LENGTH, metric_key_prefix="final")
print(f"FINAL CER: {metrics['final_cer']:.4f}")
print(f"FINAL WER: {metrics['final_wer']:.4f}")

## 9. Look at predictions

In [ ]:
model.eval()
n = 6
fig, axes = plt.subplots(n, 1, figsize=(10, 2.2 * n))
for ax, ex in zip(axes, ds["test"].select(range(n))):
    img = ex["image"].convert("RGB")
    pv = processor([img])["pixel_values"].to(model.device)
    with torch.no_grad():
        ids = model.generate(pv, num_beams=4, max_new_tokens=MAX_TARGET_LENGTH)
    pred = tokenizer.batch_decode(ids, skip_special_tokens=True)[0]
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"GT  : {ex['text']}\nPRED: {pred}", loc="left", fontsize=9)
plt.tight_layout(); plt.show()

## 10. Save the model (to Drive)

This saved folder is the **input model for stage 2** (human finetune): point the
human notebook's `MODEL_ID` at it, or push it to the Hub first.

In [ ]:
final_dir = f"{OUTPUT_DIR}/final"
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print("saved ->", final_dir)

# Durable off-Drive copy so you can warm-start later from any machine.
# Authenticate once first:  from huggingface_hub import notebook_login; notebook_login()
HUB_REPO = f"cyttic/{RUN_NAME}-pretrained"   # variant-specific repo
model.push_to_hub(HUB_REPO)
tokenizer.push_to_hub(HUB_REPO)
print("pushed ->", HUB_REPO)